# Midterm Part A – thin Colab launcher
Clones the fork, installs `midterm_a/requirements.txt` and runs `train.py` once per config. All logic lives in the repo.

Runtime → Change runtime type → **T4 GPU**, then **Run all**. Checkpoints, results, logs and the dataset cache live on Google Drive: after a disconnect just **Run all** again — finished work is skipped and interrupted runs resume from `ckpt/last.ckpt`.

Each run prints only its key lines; the full log of every run is in `ssl_midterm_a/logs/` on Drive. A failing run stops *Run all*.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
FORK_URL = "https://github.com/rafihi/representation-learning-course.git"
WORK = "/content/drive/MyDrive/ssl_midterm_a"   # runs/, results/, data_cache/, logs/ on Drive
!mkdir -p {WORK}/runs {WORK}/results {WORK}/data_cache {WORK}/logs
!rm -rf /content/repo && git clone -q --filter=blob:none --sparse {FORK_URL} /content/repo
%cd /content/repo
!git sparse-checkout set midterm_a
%cd /content/repo/midterm_a
!ln -s {WORK}/runs runs && ln -s {WORK}/results results && ln -s {WORK}/data_cache data_cache
!pip install -q -r requirements.txt 2>&1 | grep -v "dependency resolver\|requires\|incompatible" || true
!pip freeze > {WORK}/requirements.lock.txt   # exact environment (torch included): commit it as midterm_a/requirements.lock.txt
!git log -1 --format='commit %H'
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# run(config): launches `python train.py --config <config>`; the full log goes to Drive (logs/<config>.log),
# only key lines are shown here (keeps the browser responsive). A failed run raises -> "Run all" stops there.
import os, re, subprocess
from pathlib import Path
os.environ.update(PYTHONUNBUFFERED="1", HF_DATASETS_DISABLE_PROGRESS_BARS="1", TQDM_DISABLE="1")
LOGS = Path(WORK) / "logs"; LOGS.mkdir(parents=True, exist_ok=True)
KEY = re.compile(r"^\[(done|skip|from_sweep|finetune|warn|layer)|Traceback|Error|BEST|best val|^\{'dataset'|^\{'model|knn_top1 =|paired|worst classes|TEST split")

def run(config, *overrides):
    log = LOGS / (Path(config).stem + ".log")
    cmd = ["python", "train.py", "--config", config, "--set", "trainer.progress=false", *overrides]
    print("$", " ".join(cmd), " | log:", log, flush=True)
    tail = []
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            f.write(line); f.flush()
            tail = (tail + [line])[-60:]
            if KEY.search(line):
                print(line.rstrip()[:300], flush=True)
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{config} failed (exit code {rc}). Full log: {log}")
    print(f"OK  {config}\n", flush=True)


In [ ]:
# What is already on Drive (finished work is skipped automatically)
w = Path(WORK)
print("dataset cache      :", (w / "data_cache/tiny_imagenet_uint8.npz").exists())
print("A2 sweep trials    :", len(list((w / "results/a1_tune_dae").glob("trial_*/metrics.json"))), "/ 20 finished")
for r in ["dae_final", "vanilla_ae_final"]:
    d = w / "runs" / r
    print(f"{r:19}:", "finished" if (d / "metrics.json").exists() else "partial (will resume)" if (d / "ckpt/last.ckpt").exists() else "not started")
print("fine-tuning rows   :", sum(1 for _ in open(w / "results/a5_finetune/results.csv")) - 1 if (w / "results/a5_finetune/results.csv").exists() else 0, "/ 18")


## 0. Sanity check (synthetic data, ~1 min)

In [ ]:
!SSL_FAKE_DATA=1 SSL_TINY_BACKBONE=1 python -m pytest -q tests 2>&1 | tail -3

## A2 – tuning (proxy, 20 trials) + reconstruction-vs-k-NN figure
The first run also downloads Tiny-ImageNet once into the Drive cache (a few minutes).

In [ ]:
run("configs/a1_tune_dae.yaml")

## A1 / A3 – final DAE (hyper-parameters taken automatically from the best sweep trial) and the vanilla-AE baseline

In [ ]:
run("configs/a2_dae_final.yaml")

In [ ]:
run("configs/a3_vanilla_ae_final.yaml")

## A3 – frozen protocol (k-NN + linear probe) vs. baselines

In [ ]:
run("configs/a4_eval.yaml")

## A3 / A4 – fine-tuning 1/10/100 % + label-efficiency figure

In [ ]:
run("configs/a5_finetune.yaml")

## A4 – corruption curves (k-NN + linear probe, + bonus `patch_shuffle`)

In [ ]:
run("configs/a6_robustness.yaml")

## A5 (+A1, A2) – per-class accuracy, t-SNE, retrieval, layer choice, reconstructions

In [ ]:
run("configs/a7_interpret.yaml")

## Final numbers on the TEST split (once; every choice above was made on validation)

In [ ]:
for c in ["a4_eval_test", "a5_finetune_test", "a6_robustness_test", "a7_interpret_test"]:
    run(f"configs/{c}.yaml")

## Logs

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs